In [34]:
import os
import certifi
import requests
from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

from langchain_classic import hub



In [35]:
from langchain_classic.agents import create_react_agent,AgentExecutor


In [36]:
os.environ["SSL_CERT_FILE"] = certifi.where()   # ON WINDOWS FOR PATH RELATED ISSUES. IT IS OPTIONAL

load_dotenv()
GOOGLEPRO_API_KEY = os.getenv("GEMINIPRO_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [37]:
search_tool = TavilySearch(max_results=2)


In [38]:
@tool
def get_weather(city: str) -> str:
    """ Get the current weather for a given city using the Weatherstack API.
    Args:
        city (str): The name of the city to get the weather for.
    Returns:
        str: The current weather for the specified city."""
        
    url = f"http://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}"
    response = requests.get(url)    
    data = response.json()
    return data["current"]["weather_descriptions"][0]       


In [39]:
result=search_tool.invoke("give me the senalte result of abdul in us today")
result

{'query': 'give me the senalte result of abdul in us today',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.clickondetroit.com/news/local/2026/11/04/abdul-el-sayed-wins-michigan-us-senate-race-ap-projects',
   'title': 'Abdul El-Sayed wins Michigan U.S. Senate race, AP projects',
   'content': 'Abdul El-Sayed is expected to win the Michigan U.S. Senate race, the Associated Press projects.\n\nHere’s a look at the results:\n\nRecommended Videos\n\n### U.S. Senate - Michigan\n\nCandidate\n\nVotes\n\n%\n\nAbdul El-Sayed\n\n#### Abdul El-Sayed(D)\n\n2,192,51952%\n\nMike Rogers\n\n#### Mike Rogers(R)\n\n1,949,84146%\n\nLydia Christensen\n\n#### Lydia Christensen(L)\n\n43,9081%\n\nTim Long\n\n#### Tim Long(U)\n\n23,2531%\n\nDouglas Marsh\n\n#### Douglas Marsh(G)\n\n10,9380%\n\nWalter Kristy\n\n#### Walter Kristy(N)\n\n6,3870%',
   'score': 0.6864757,
   'raw_content': None,
   'id': '3f0048-00'},
  {'url': 'https://www.uppermichiganssource.com/

In [40]:
# ==========
# LLM
# =============
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash",temperature=0.2,max_output_tokens=1024,
    api_key=GOOGLEPRO_API_KEY)


In [ ]:
# llm.invoke("what is the current year and day")

In [ ]:
from langchain_core.prompts import PromptTemplate

# ========

# PROMPT
#  prompt=hub.pull("hwchase17/react")    deprecated not working
# =========
prompt = PromptTemplate.from_template(
    """Answer the question using the available tools.

You can use these tools:
{tools}

Use this format:
Question: the question to answer
Thought: your reasoning
Action: one of [{tool_names}]
Action Input: the input to the action
Observation: the tool result
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the answer

Question: {input}
Thought:{agent_scratchpad}"""
)

In [ ]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, template='Answer the question using the available tools.\n\nYou can use these tools:\n{tools}\n\nUse this format:\nQuestion: the question to answer\nThought: your reasoning\nAction: one of [{tool_names}]\nAction Input: the input to the action\nObservation: the tool result\n... (repeat Thought/Action/Action Input/Observation as needed)\nThought: I now know the final answer\nFinal Answer: the answer\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [42]:
tools =[search_tool,get_weather]

In [ ]:
# =====
#  create agent
# =====

agent =create_react_agent(llm=llm,tools=tools,prompt=prompt)

In [ ]:
# ====
# Agent Executor
# ====

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


In [ ]:
# ========
#  RUN AGENT EXECUTOR
# ========

response = agent_executor.invoke({
    "input":"give me todays senate result of abdul in us today and give the weather of michigan today"
    })


In [ ]:
print(response["output"])